# B03 — *E. coli* 단일세포 성장률과 분열 관련 시간 분석

**BioCulture | Dunlop Lab CTC 실제 데이터 | 교육·검증 통합본 Clean v2**

**목표:** B02에서 확인한 `TRA`의 지속 Cell ID를 전체 시퀀스에 적용하여 세포별 면적·장축 시계열을 구축하고, 길이·면적의 로그 선형 성장률을 추정하며, 계보표의 부모–자식 기록에서 분열 관련 시간 간격을 계산합니다.

**해석 주의:** `major_axis_length`는 픽셀 기준 장축이지 실제 세포 길이의 직접 측정값이 아닙니다. 모형 적합은 관측 구간의 기술적 요약이지 생물학적 성장법칙의 입증이 아닙니다. `man_track.txt`의 `start/end`만으로 세대시간을 확정하지 않습니다. CTC의 추적 정답은 학습용 기준이며 실제 동역학 측정의 편향·오류가 있을 수 있습니다.

**입력:** `data/bioculture/CTC.zip` / `CTC/train/69`, `69_GT/TRA` / 시간 간격 5분 (데이터셋 README)  
**출력:** `results/bioculture/B03/`의 CSV·PNG. **필요 라이브러리**: numpy, pandas, matplotlib, tifffile, imagecodecs (LZW), scikit-image, scipy. **Kernel:** Python (BioImage AI).

**버전 정책:** v1의 기존 분석 8단계를 그대로 보존하고, 성장률 품질 감사·세대시간 후보의 검열 구분·결과 재현성 검사를 덧붙였다. 최신 코드로 다시 실행하기 전에는 기존 저장 출력과 혼용하지 않는다.

## 1. 환경·프로젝트 루트·데이터 검사
코드 셀은 **입력 → 처리 → 출력**을 명시하는 주석을 포함합니다. 프로젝트 루트 또는 `notebooks` 폴더에서 실행할 수 있으며, ZIP 경로가 다르면 `ZIP_PATH`를 직접 지정할 수 있습니다.

In [ ]:
# [B03-01] 라이브러리, 입력 파일, 결과 폴더 준비
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import re, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from skimage.measure import regionprops_table
from scipy.stats import linregress

# 데이터 경로: Jupyter 현재 위치가 프로젝트 루트이거나 notebooks 하위라고 가정
choices = [Path.cwd()/"data"/"bioculture"/"CTC.zip",
           Path.cwd().parent/"data"/"bioculture"/"CTC.zip"]
ZIP_PATH = next((x.resolve() for x in choices if x.is_file()), None)
if ZIP_PATH is None:
    raise FileNotFoundError("bioimage-ai/data/bioculture/CTC.zip 경로를 확인하세요.")
PROJECT_ROOT = ZIP_PATH.parent.parent.parent
RESULTS_DIR = PROJECT_ROOT / "results" / "bioculture" / "B03"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SPLIT, SEQUENCE = "train", "69"
FRAME_INTERVAL_MIN = 5.0
# 실제 데이터 전체 193프레임을 기본으로 검사; 개발시 정수로 줄일 수 있음
MAX_FRAMES = None
print("Python:", sys.executable)
print("CTC:", ZIP_PATH)
print("Results:", RESULTS_DIR)

## 2. 원본/TRA 프레임 대응 및 계보표 감사
B03의 추적 특성은 **SEG가 아니라 TRA 레이블**로 계산합니다. 원본 TIFF와 TRA 사이의 프레임 번호를 검사하고, 계보표의 유일성·프레임 범위·부모 참조를 확인합니다. 판정에 실패하면 분석을 중단합니다.

In [ ]:
# [B03-02] 파일명에서 프레임 번호를 추출하고 동일 시점 TIFF를 정확히 매칭
img_prefix = f"CTC/{SPLIT}/{SEQUENCE}/"
tra_prefix = f"CTC/{SPLIT}/{SEQUENCE}_GT/TRA/"
track_file = tra_prefix + "man_track.txt"

def frame_number(name, kind):
    pattern = r"/t(\d+)\.tif$" if kind == "image" else r"/man_track(\d+)\.tif$"
    match = re.search(pattern, name, re.I)
    return int(match.group(1)) if match else None

def read_tif(z, name):
    # ZIP 압축을 풀지 않고 내부 TIFF를 메모리에서 열기
    return tifffile.imread(BytesIO(z.read(name)))

with ZipFile(ZIP_PATH) as z:
    files = z.namelist()
    img_paths = [x for x in files if x.startswith(img_prefix) and frame_number(x,"image") is not None]
    tra_paths = [x for x in files if x.startswith(tra_prefix) and frame_number(x,"tra") is not None]
    assert track_file in files, "man_track.txt를 찾을 수 없습니다."
    tracks = pd.read_csv(BytesIO(z.read(track_file)), sep=r"\s+", header=None,
                         names=["cell_id", "start", "end", "parent_id"])
image_by_t = {frame_number(x,"image"):x for x in img_paths}
tra_by_t = {frame_number(x,"tra"):x for x in tra_paths}
assert len(image_by_t)==len(img_paths) and len(tra_by_t)==len(tra_paths), "프레임 번호 중복"
all_frames = sorted(set(image_by_t)&set(tra_by_t))
assert all_frames and set(image_by_t)==set(tra_by_t), "영상과 TRA의 프레임 불일치"
frames = all_frames if MAX_FRAMES is None else all_frames[:int(MAX_FRAMES)]
assert tracks.cell_id.is_unique and (tracks.cell_id>0).all(), "중복 또는 잘못된 Cell ID"
assert (tracks.start<=tracks.end).all(), "추적 시작/끝 시점 오류"
assert (tracks.start>=min(all_frames)).all() and (tracks.end<=max(all_frames)).all(), "기간 범위 오류"
known_ids = set(map(int,tracks.cell_id))
assert set(map(int,tracks.parent_id))-{0} <= known_ids, "존재하지 않는 부모 ID"
assert not (tracks.cell_id==tracks.parent_id).any(), "자기 자신을 부모로 참조"
print("Images/TRA matched:",len(all_frames),"| selected:",len(frames))
print("Track records:",len(tracks),"| parent-linked:",int((tracks.parent_id!=0).sum()))
display(tracks.head(8))
tracks.to_csv(RESULTS_DIR/"track_metadata.csv",index=False)

## 3. 전체 프레임 TRA 품질 검사 및 세포별 형태 측정
마스크에 관찰된 ID가 추적표의 해당 기간에 포함되는지 검사합니다. **기간에 속해야 하는 ID**가 마스크에 없거나 예기치 않은 ID가 있다면 기록합니다. 실제 주석 규칙에 따라 일부 예외가 있을 수 있으므로, 문제가 발생하면 원본 확인 전 성장률 해석을 보류합니다.

측정값: 면적 `area_px2`, 장축 `major_axis_px`, 단축 `minor_axis_px`, 형태 비율. 프레임×Cell ID별 1행으로 보관합니다.

In [ ]:
# [B03-03] 모든 선택 프레임을 검사하며 TRA 개별 ID의 형태를 측정
properties=("label", "area", "major_axis_length", "minor_axis_length", "centroid")
qc_rows=[]
measured=[]
for_idx=tracks.set_index("cell_id")
with ZipFile(ZIP_PATH) as z:
    for t in frames:
        img=read_tif(z,image_by_t[t])
        tra=read_tif(z,tra_by_t[t])
        # 원본 RGB는 H×W×3, TRA는 H×W이므로 공간 크기만 비교
        spatial=img.shape[:2] if img.ndim==3 and img.shape[-1] in (3,4) else img.shape
        if tra.ndim!=2 or not np.issubdtype(tra.dtype,np.integer) or spatial!=tra.shape:
            raise ValueError(f"Frame {t}: TIFF/TRA 형태 불일치 {img.shape} {tra.shape}")
        seen=set(map(int,np.unique(tra)))-{0}
        expected=set(map(int,tracks.loc[(tracks.start<=t)&(tracks.end>=t),"cell_id"]))
        qc_rows.append(dict(frame=t, observed_count=len(seen), expected_count=len(expected),
                            missing_ids=",".join(map(str,sorted(expected-seen))),
                            unexpected_ids=",".join(map(str,sorted(seen-expected))),
                            missing_count=len(expected-seen), unexpected_count=len(seen-expected)))
        if not seen:
            continue
        d=pd.DataFrame(regionprops_table(tra,properties=properties))
        d=d.rename(columns={"label":"cell_id","area":"area_px2",
                            "major_axis_length":"major_axis_px","minor_axis_length":"minor_axis_px"})
        d.insert(0,"frame",t)
        d["time_min"]=(t-min(all_frames))*FRAME_INTERVAL_MIN
        d["aspect_ratio"]=d.major_axis_px/d.minor_axis_px.replace(0,np.nan)
        measured.append(d)
qc=pd.DataFrame(qc_rows)
if not measured: raise RuntimeError("TRA에서 관측된 세포가 없습니다.")
obs=pd.concat(measured,ignore_index=True).sort_values(["cell_id","frame"]).reset_index(drop=True)
assert not obs.duplicated(["cell_id","frame"]).any(), "동일 Cell ID-프레임 중복"
qc.to_csv(RESULTS_DIR/"frame_id_qc.csv",index=False)
obs.to_csv(RESULTS_DIR/"track_measurements.csv",index=False)
print("QC frames:",len(qc),"| missing frames:",int((qc.missing_count>0).sum()),
      "| unexpected frames:",int((qc.unexpected_count>0).sum()))
print("Measurements:",len(obs),"| unique tracks observed:",obs.cell_id.nunique())
display(qc.head());display(obs.head())

## 4. 성장률 추정 기준 설정
길이나 면적 측정값 자체가 양수일 때 `log(size) = intercept + μ × time` 모델을 최소제곱 적합합니다. **연속 프레임**, **최소 관측 개수**, **추적표 기간 내 측정**, **양의 측정값**을 요구합니다. 끝까지 관측되었다고 해서 정상적인 세대 전체가 관찰되었다는 뜻은 아닙니다.

`μ`는 분당 로그 변화율(`min⁻¹`), `doubling_time = ln(2)/μ`는 **성장모델상의 배가시간**이며 **분열까지 걸린 시간과 구분**합니다. 음의 기울기를 포함해 계산하되 `μ≤0`의 배가시간은 결측 처리합니다.

In [ ]:
# [B03-04] 성장률 적합 함수와 품질 기준
MIN_POINTS=4
MIN_R2=0.70  # 자동 필터에서 제외 기준이 아닌 참고용 품질 플래그

def fit_growth(group, value_column):
    # 각 추적 ID의 원래 관측을 시간 순으로 정렬
    data=group.sort_values("frame").dropna(subset=[value_column]).copy()
    data=data.loc[np.isfinite(data[value_column]) & (data[value_column]>0)]
    if len(data)<MIN_POINTS or not np.all(np.diff(data.frame.to_numpy())==1):
        return None
    # 실제 측정값을 로그로 변환하여 기울기를 추정
    reg=linregress(data.time_min.to_numpy(),np.log(data[value_column].to_numpy()))
    mu=float(reg.slope)
    return dict(n_points=len(data),first_frame=int(data.frame.min()),last_frame=int(data.frame.max()),
                observed_duration_min=float(data.time_min.max()-data.time_min.min()),
                mu_per_min=mu,intercept=float(reg.intercept),r2=float(reg.rvalue**2),
                doubling_time_min=float(np.log(2)/mu) if mu>0 else np.nan,
                fit_quality="R2>=threshold" if reg.rvalue**2>=MIN_R2 else "low_R2")

fits=[]
for cell_id,group in obs.groupby("cell_id"):
    for variable in ("major_axis_px","area_px2"):
        outcome=fit_growth(group,variable)
        if outcome is not None:
            fits.append(dict(cell_id=int(cell_id),metric=variable,**outcome))
growth=pd.DataFrame(fits)
if growth.empty:
    raise RuntimeError("성장률 적합 기준을 만족하는 Cell ID가 없습니다. 기준과 영상 범위를 점검하세요.")
growth.to_csv(RESULTS_DIR/"growth_rate_fits.csv",index=False)
print("Fit rows:",len(growth),"| IDs:",growth.cell_id.nunique())
display(growth.head(10))

## 5. 실제 시계열과 적합곡선 비교
예시 추적 중 하나의 장축 관측값과 로그 선형 모델 예측을 겹쳐 그립니다. 잔차 패턴과 비단조 변화를 살피고, **높은 R²만으로 모델이 올바르다고 결론내리지 않습니다.**

In [ ]:
# [B03-05] 실제 길이 관측값과 모델 적합 결과 시각화
length_fits=growth.loc[growth.metric=="major_axis_px"].sort_values(["n_points","r2"],ascending=False)
assert not length_fits.empty
example=length_fits.iloc[0]
cell_id=int(example.cell_id)
a=obs.loc[obs.cell_id==cell_id].sort_values("frame")
pred=np.exp(float(example.intercept)+float(example.mu_per_min)*a.time_min)
fig,ax=plt.subplots(figsize=(8,4.5))
ax.plot(a.time_min,a.major_axis_px,"o-",label="Observed major axis (px)")
ax.plot(a.time_min,pred,"--",label="Log-linear fit")
ax.set(xlabel="Time since first sequence frame (min)",ylabel="Major axis (px)",
       title=f"Cell {cell_id}: growth fit, R²={example.r2:.2f}")
ax.legend();ax.grid(alpha=.25)
plt.tight_layout();plt.savefig(RESULTS_DIR/"example_growth_fit.png",dpi=160);plt.show()

## 6. 부모–자식 관계의 시간 간격
계보표에서 같은 부모를 참조하는 자식들이 있는지 확인합니다. 두 자식이 연결된 부모는 **분열 관련 후보 사건**으로 취급합니다. 계보 주석의 부모 마지막 프레임과 자식 최초 프레임을 비교합니다. **이 간격은 세대시간이 아니며**, 부모나 자손의 전체 생애가 관측된 것인지 추가 구별이 필요합니다.

In [ ]:
# [B03-06] 계보를 이용한 분열 관련 후보 사건 및 프레임 간격 표
child=tracks.loc[tracks.parent_id!=0].copy()
rows=[]
for parent_id,g in child.groupby("parent_id"):
    parent=for_idx.loc[int(parent_id)]
    rows.append(dict(parent_id=int(parent_id),n_children=len(g),
                     child_ids=",".join(map(str,sorted(g.cell_id.astype(int)))),
                     parent_start=int(parent.start),parent_end=int(parent.end),
                     earliest_child_start=int(g.start.min()),latest_child_start=int(g.start.max()),
                     parent_end_to_first_child_min=float((g.start.min()-parent.end)*FRAME_INTERVAL_MIN),
                     child_starts_same_frame=bool(g.start.nunique()==1),
                     two_children=bool(len(g)==2)))
events=pd.DataFrame(rows)
events["candidate_division"] = events.two_children & events.child_starts_same_frame & (events.earliest_child_start==events.parent_end+1)
events.to_csv(RESULTS_DIR/"lineage_candidate_events.csv",index=False)
print("Parents with children:",len(events),"| two children:",int(events.two_children.sum()),
      "| consecutive-frame candidate events:",int(events.candidate_division.sum()))
display(events.head(12))

## 7. 관찰된 추적 지속시간과 세대시간의 차이
각 TRA 레코드의 `(end-start)*5`분은 **관측된 추적 구간의 첫·마지막 시점 간 길이**입니다. 세대시간으로 단정하지 않고 `observed_track_span_min`으로 저장합니다. 양 끝이 관찰창 경계에 닿으면 관찰 기간이 잘린(censored) 구간일 수 있습니다.

분열 후보 부모의 시계열을 검토하고 별도의 시각·주석 규칙 검증이 끝난 후에야 세대시간 분석으로 확장합니다.

In [ ]:
# [B03-07] 검열 가능성을 포함한 추적 구간의 시간 요약
spans=tracks.copy()
spans["observed_track_span_min"]=(spans.end-spans.start)*FRAME_INTERVAL_MIN
spans["touches_sequence_start"]=spans.start.eq(min(all_frames))
spans["touches_sequence_end"]=spans.end.eq(max(all_frames))
spans["has_annotated_parent"]=spans.parent_id.ne(0)
spans["has_annotated_children"]=spans.cell_id.isin(events.parent_id)
spans.to_csv(RESULTS_DIR/"track_observation_spans.csv",index=False)
print("Touches start:",int(spans.touches_sequence_start.sum()),
      "| touches end:",int(spans.touches_sequence_end.sum()))
display(spans[["cell_id","observed_track_span_min","touches_sequence_start",
               "touches_sequence_end","has_annotated_children"]].head())

## 8. 분석 결과의 기본 요약과 저장 확인
성장률 적합의 수, 높은 설명력을 보인 적합의 수, 계보 후보 수를 출력합니다. 이들은 **알고리즘의 산출물**이며 아직 전체 집단의 생물학적 평균 성장률·세대시간이라고 해석하지 않습니다.

In [ ]:
# [B03-08] 재현 가능한 출력 파일과 주요 품질 통계 정리
report=pd.DataFrame([{
    "dataset":"CTC/train/69", "frames_total":len(all_frames),"frames_analyzed":len(frames),
    "track_records":len(tracks),"cell_observations":len(obs),
    "tracks_observed":obs.cell_id.nunique(),"frames_with_missing_ids":int((qc.missing_count>0).sum()),
    "frames_with_unexpected_ids":int((qc.unexpected_count>0).sum()),
    "growth_fit_rows":len(growth),"fits_r2_ge_threshold":int((growth.r2>=MIN_R2).sum()),
    "candidate_division_events":int(events.candidate_division.sum()),
    "time_step_minutes":FRAME_INTERVAL_MIN,
}])
report.to_csv(RESULTS_DIR/"b03_run_summary.csv",index=False)
display(report)
print("Saved outputs:")
for f in sorted(RESULTS_DIR.iterdir()):print(" -",f.name)

## 9. 성장률 품질 감사: 음수 기울기, 짧은 관찰 구간, 길이·면적 비교
기존 B03의 모든 적합 결과를 보존하되, **좋은 수학적 적합과 믿을 만한 생물학적 추정은 별개**임을 데이터로 점검한다. `R²≥0.70`은 교육용 참고 임계값이지 검증된 품질 표준이 아니다. 또한 양수 `μ`에만 정의되는 배가시간을 분열 주기로 해석하지 않는다.

In [ ]:
# [B03-09] 성장률 적합의 방향과 품질 분류; 판정 근거를 CSV에 보존
quality = growth.copy()
quality["slope_direction"] = np.select(
    [quality.mu_per_min > 0, quality.mu_per_min < 0],
    ["positive", "negative"], default="zero")
quality["r2_pass_reference"] = quality.r2.ge(MIN_R2)
quality["short_window_warning"] = quality.n_points.lt(6)
# 양수 성장률과 높은 R²를 동시에 만족해도 생물학적 검증 완료가 아님
quality["exploratory_positive_fit"] = (
    quality.slope_direction.eq("positive") & quality.r2_pass_reference
)
quality.to_csv(RESULTS_DIR / "growth_quality_audit.csv", index=False)
quality_summary = quality.groupby(["metric", "slope_direction"], dropna=False).agg(
    fit_count=("cell_id", "size"),
    median_r2=("r2", "median"),
    median_duration_min=("observed_duration_min", "median"),
    above_reference_r2=("r2_pass_reference", "sum")
).reset_index()
display(quality_summary)
print("Quality audit rows:", len(quality), "| Short windows (<6 observations):", int(quality.short_window_warning.sum()))

## 10. 장축·면적 성장률 비교
두 지표는 서로 다른 물리량이다. 지수 모델의 기울기가 같을 것이라고 가정하지 않는다. 동일 Cell ID에 대해 두 성장률을 나란히 놓고 차이의 분포를 살펴본다.

In [ ]:
# [B03-10] 한 Cell ID에서 두 지표의 성장률을 대응시켜 비교
paired = growth.pivot(index="cell_id", columns="metric", values="mu_per_min")
paired = paired.rename(columns={"major_axis_px": "mu_length_per_min", "area_px2": "mu_area_per_min"})
paired = paired.dropna(subset=["mu_length_per_min", "mu_area_per_min"]).copy()
paired["mu_area_minus_length"] = paired.mu_area_per_min - paired.mu_length_per_min
paired.to_csv(RESULTS_DIR / "growth_length_area_comparison.csv")
print("Paired tracks:", len(paired))
display(paired.head())
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(paired.mu_length_per_min, paired.mu_area_per_min, s=12, alpha=0.65)
ax.axhline(0, linewidth=0.6); ax.axvline(0, linewidth=0.6)
ax.set(xlabel="Length-based log slope (/min)", ylabel="Area-based log slope (/min)",
       title="Within-track comparison of growth slopes")
plt.tight_layout()
fig.savefig(RESULTS_DIR / "growth_length_area_comparison.png", dpi=160)
plt.show()

## 11. 출생과 분열이 모두 기록된 세포의 계보 기반 시간 구간
계보상 **부모가 기록된 세포**에서 출생 시점을 자식의 시작 프레임으로 근사하고, 그 세포 자신에게 **두 자식이 연속 프레임에 등장**하는 경우를 다음 분열의 후보로 삼는다. 이 경우 `자식 시작 프레임 − 본인 시작 프레임`에 5분을 곱한 값을 구할 수 있다. 다만 이는 **계보 주석 기준 세대시간 후보(interval)**이지 현미경 관찰로 독립 확인된 세대시간이 아니다. 영상 경계에 걸린 추적이나 누락 프레임이 없는지 별도 검사를 병행한다.

In [ ]:
# [B03-11] 계보 기준 양쪽 사건을 가진 세포만 선별
confirmed_parent_links = set(tracks.loc[tracks.parent_id.ne(0), "cell_id"].astype(int))
division_parent_ids = set(events.loc[events.candidate_division, "parent_id"].astype(int))
complete_ids = confirmed_parent_links & division_parent_ids
intervals = tracks.loc[tracks.cell_id.isin(complete_ids)].copy()
intervals = intervals.merge(
    events[["parent_id", "earliest_child_start", "latest_child_start"]],
    left_on="cell_id", right_on="parent_id", how="left", validate="one_to_one",
    suffixes=("", "_event"))
intervals["lineage_generation_interval_min"] = (
    intervals.earliest_child_start - intervals.start
) * FRAME_INTERVAL_MIN
intervals["left_boundary_censored"] = intervals.start.eq(min(all_frames))
intervals["right_boundary_censored"] = intervals.end.eq(max(all_frames))
intervals["interval_valid"] = (
    intervals.lineage_generation_interval_min.gt(0)
    & intervals.earliest_child_start.eq(intervals.end + 1)
    & ~intervals.left_boundary_censored
    & ~intervals.right_boundary_censored
)
intervals.to_csv(RESULTS_DIR / "lineage_generation_intervals.csv", index=False)
print("Both-end lineage records:", len(intervals))
print("Valid candidate intervals:", int(intervals.interval_valid.sum()))
if intervals.interval_valid.any():
    display(intervals.loc[intervals.interval_valid, [
        "cell_id", "start", "end", "earliest_child_start", "lineage_generation_interval_min"
    ]].head(12))
else:
    print("엄격한 기준을 만족하는 세대시간 후보가 없습니다.")

## 12. 통합 품질 점검 및 실행 파일 검증
**핵심 조건**은 전체 프레임에서 예상 TRA 레이블과 관찰 레이블의 누락·초과가 0인지이다. 실제 데이터에서 새 결과가 저장되었는지 점검한다. 기존 193/247/1454/404/120은 사용자 실행 결과이며 코드의 정답 상수로 강제하지 않는다.

In [ ]:
# [B03-12] 출력 검증 — 실행 실패 시 경고 또는 예외로 구분
required_outputs = [
    "track_metadata.csv", "frame_id_qc.csv", "track_measurements.csv",
    "growth_rate_fits.csv", "lineage_candidate_events.csv", "track_observation_spans.csv",
    "b03_run_summary.csv", "growth_quality_audit.csv",
    "growth_length_area_comparison.csv", "lineage_generation_intervals.csv"
]
missing_files = [name for name in required_outputs if not (RESULTS_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"생성되지 않은 결과 파일: {missing_files}")
if MAX_FRAMES is None:
    assert not qc.missing_count.any() and not qc.unexpected_count.any(), (
        "TRA 프레임별 ID 일치 검사를 통과하지 못했습니다. frame_id_qc.csv를 확인하세요."
    )
print("Output files verified:", len(required_outputs))
print("Frame QC totals:", int(qc.missing_count.sum()), int(qc.unexpected_count.sum()))
print("Important: a candidate lineage interval is not independent biological confirmation.")

## 9. 해석의 범위 · 품질 검증 · 연습문제

**이 Notebook이 제공하는 것:** (1) 전체 시퀀스의 TRA ID 감사, (2) 세포별 면적·장축 시계열, (3) 로그 선형 모형의 기간 한정 성장률과 설명력, (4) 부모–자식 관계로부터 분열 후보와 관찰 기간 지표.

**이 Notebook만으로 확정하지 않는 것:** 모든 추적의 완전성, 실제 µm 단위 세포 길이, 확정된 단일세포 세대시간, 실제 세포 생존성 또는 CFU. 특히 `doubling_time_min`은 **지수 성장 모델의 배가시간**이지 **생물학적 분열 주기**가 아닙니다.

**품질 검사**: `frame_id_qc.csv` 전체의 `missing_count` 및 `unexpected_count` 확인, R²가 낮은 추적의 직접 이미지 검토, 분열 후보의 부모/자식 TRA 이미지 육안 검사.

**연습문제:** (1) `MIN_POINTS`를 4에서 6으로 바꿔 유효 적합 수 비교 (2) 면적 성장률과 장축 성장률의 상관 분석 (3) `candidate_division=True`인 계보 3개의 실제 TRA 영상 확인 (4) 관측 구간이 전체 생애를 대표하지 못하는 이유 설명.

**다음 연구 단계:** B04에서 세포 간 변동성, 적합법의 비교, 생물학적 품질 기준과 불확실성 분석. `README`의 수치 확인 전 실제 결과를 확정적으로 보고하지 않습니다.

**v2 연습문제:** (1) `growth_quality_audit.csv`로 음수 기울기 비율을 계산한다. (2) 길이·면적 적합 결과의 부호가 달라지는 Cell ID를 찾는다. (3) `lineage_generation_intervals.csv`에서 검열 조건이 어떤 후보를 제외하는지 설명한다. (4) 세대시간 후보의 분포와 지수모델 배가시간 분포가 서로 다른 이유를 토론한다. **모범답안은 대응 Word 실습 해설서에 수록할 예정이다.**